<a href="https://colab.research.google.com/github/vitoriaferreirap/DeepLearning/blob/main/PoseEstimation%20_IC/AdptacaoDominio/AdpatacaoDominio_YOLOv11_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import zipfile
# Descompacta pastas raiz colab
caminho_zip = '/content/dados.zip'
pasta_destino = '/content/'

# Percorrendo hierarquia de pastas
with zipfile.ZipFile(caminho_zip, 'r') as zip_ref:
    zip_ref.extractall(pasta_destino)

# DataFrame do Pandas (pd.DataFrame)
 - Estrutura de dados bidimensional.
 - **(read_hdf)** Responsavel por transformar esse formato binário em uma tabela, organizando os dados em linhas e colunas rotuladas, onde cada coluna pode conter um tipo diferente de dado.
 - Ferramenta mais utilizada em Python para análise e manipulação de dados.
 - Permite importar arquivos (como CSV e Excel), tratar dados ausentes, filtrar informações e fazer cálculos estatísticos complexos rapidamente.
  - **Nota:** pode ser considerado uma interface de manipulação. Sem ele teria que lidar com os bytes brutos do arquivo .h5, usando-o, tem uma tabela onde cada linha é um frame e cada coluna é uma coordenação (X, Y).
 # HDF5 (O Formato de Armazenamento):
 - Formato de arquivo hierárquico (como se fosse um sistema de pastas dentro de um único arquivo) que guarda números brutos de forma muito compacta. É excelente para salvar gigabytes de dados de vídeo.
- Formato binário eficiente para grandes conjuntos de dados científicos, muito usado pelo DeepLabCut.
- O parâmetro **key='df_with_missing'** acessa a "tabela" específica dentro desse arquivo HDF5 que contém os dados/coordenadas das articulações (incluindo possíveis valores faltantes).

In [ ]:
import os
import pandas as pd

# Caminho raiz para arquivos H5 já com keypoints criticos filtrados
pasta_resultados = '/content/dados/resultados_predicao_KeypointsCriticosFiltrados'

# Ex: Acessando o primeiro arquivo da pasta Resultado_A
pasta_alvo = os.path.join(pasta_resultados, 'Resultado_A')
arquivos = [f for f in os.listdir(pasta_alvo) if f.endswith('.h5')]

if arquivos:
    caminho_arquivo = os.path.join(pasta_alvo, arquivos[0])
    df = pd.read_hdf(caminho_arquivo, key='df_with_missing')

    print(f"Lendo com sucesso: {arquivos[0]}")
    print(df.head())
else:
    print("Nenhum arquivo .h5 encontrado nesta pasta.")

Lendo com sucesso: 8_superanimal_quadruped_snapshot-hrnet_w32-004_snapshot-fasterrcnn_resnet50_fpn_v2-004.h5
scorer      superanimal_quadruped_snapshot-hrnet_w32-004_snapshot-fasterrcnn_resnet50_fpn_v2-004  \
individuals                                                                              animal0   
bodyparts                                                                                   nose   
coords                                                                                         x   
0                                                   987.609375                                     
1                                                   956.835938                                     
2                                                   914.960938                                     
3                                                   887.375000                                     
4                                                   838.492188                             

# Filtragem Keypoints Críticos
- filtragem substitutiva (sobrescrevendo os arquivos .h5 na pasta resultados_predicao_KeypointsCriticosFiltrados com apenas os 15 pontos importantes).


In [ ]:
import pandas as pd
import os

pontos_criticos = [
    'front_left_knee', 'front_left_paw', 'front_right_knee', 'front_right_paw',
    'back_left_knee', 'back_left_paw', 'back_right_knee', 'back_right_paw',
    'back_base', 'back_middle', 'back_end', 'neck_base', 'neck_end', 'nose', 'belly_bottom'
]

pasta_alvo = '/content/dados/resultados_predicao_KeypointsCriticosFiltrados'
subpastas = ['Resultado_A', 'Resultado_C', 'Resultado_D']

for sub in subpastas:
    caminho_sub = os.path.join(pasta_alvo, sub)
    for arquivo in os.listdir(caminho_sub):
        if arquivo.endswith('.h5'):
            caminho_completo = os.path.join(caminho_sub, arquivo)
            df = pd.read_hdf(caminho_completo, key='df_with_missing')

            # Filtro rigoroso: mantém apenas as colunas que estão na nossa lista
            colunas_selecionadas = [c for c in df.columns if c[2] in pontos_criticos]
            df_filtrado = df[colunas_selecionadas]

            # Salva sobrescrevendo o arquivo original com o conteúdo reduzido
            df_filtrado.to_hdf(caminho_completo, key='df_with_missing', format='table')

print("Filtro aplicado! Agora todos os H5 contêm apenas os 15 pontos críticos.")

Filtro aplicado! Agora todos os H5 contêm apenas os 15 pontos críticos.


In [ ]:
# Teste de conferência de keypoints em um dos arquivos
caminho_teste = '/content/dados/resultados_predicao_KeypointsCriticosFiltrados/Resultado_A/1_superanimal_quadruped_snapshot-hrnet_w32-004_snapshot-fasterrcnn_resnet50_fpn_v2-004.h5'

df_check = pd.read_hdf(caminho_teste, key='df_with_missing')
pontos_presentes = df_check.columns.get_level_values('bodyparts').unique()

print(f"Total de pontos encontrados agora: {len(pontos_presentes)}")
print("Lista de pontos confirmada:", list(pontos_presentes))

Total de pontos encontrados agora: 15
Lista de pontos confirmada: ['nose', 'neck_base', 'neck_end', 'back_base', 'back_end', 'back_middle', 'front_left_knee', 'front_left_paw', 'front_right_knee', 'front_right_paw', 'back_left_paw', 'back_left_knee', 'back_right_knee', 'back_right_paw', 'belly_bottom']


# arquivos .txt
- estrutura exata (classe 0 seguida pelas 15 coordenadas normalizadas x, y) pronta para ser lida por qualquer modelo YOLO de detecção de pose.
- Para garantir que os .txt estão corretos, vamos verificar o número de coordenadas em cada arquivo. Cada .txt deve conter exatamente 31 números (o ID da classe 0 + 15 pontos com x e y cada, totalizando 30 coordenadas + o ID).

In [ ]:
import os

pasta_coordenadas = '/content/coordenadas_transformadas_yolo'
subpastas = ['Resultado_A', 'Resultado_C', 'Resultado_D']

for sub in subpastas:
    caminho_sub = os.path.join(pasta_coordenadas, sub)
    print(f"\n--- Validando Pasta: {sub} ---")

    for arquivo in os.listdir(caminho_sub):
        if arquivo.endswith('.txt'):
            with open(os.path.join(caminho_sub, arquivo), 'r') as f:
                conteudo = f.read().split()
                # O formato é: ID(1) + 15 pontos * 2 (x,y) = 31 elementos
                if len(conteudo) == 31:
                    status = "OK (31 elementos)"
                else:
                    status = f"ERRO (Total: {len(conteudo)} elementos)"
                print(f"{arquivo}: {status}")


--- Validando Pasta: Resultado_A ---
4_superanimal_quadruped_snapshot-hrnet_w32-004_snapshot-fasterrcnn_resnet50_fpn_v2-004.txt: OK (31 elementos)
7_superanimal_quadruped_snapshot-hrnet_w32-004_snapshot-fasterrcnn_resnet50_fpn_v2-004.txt: OK (31 elementos)
3_superanimal_quadruped_snapshot-hrnet_w32-004_snapshot-fasterrcnn_resnet50_fpn_v2-004.txt: OK (31 elementos)
19_superanimal_quadruped_snapshot-hrnet_w32-004_snapshot-fasterrcnn_resnet50_fpn_v2-004.txt: OK (31 elementos)
6_superanimal_quadruped_snapshot-hrnet_w32-004_snapshot-fasterrcnn_resnet50_fpn_v2-004.txt: OK (31 elementos)
1_superanimal_quadruped_snapshot-hrnet_w32-004_snapshot-fasterrcnn_resnet50_fpn_v2-004.txt: OK (31 elementos)
16_superanimal_quadruped_snapshot-hrnet_w32-004_snapshot-fasterrcnn_resnet50_fpn_v2-004.txt: OK (31 elementos)
9_superanimal_quadruped_snapshot-hrnet_w32-004_snapshot-fasterrcnn_resnet50_fpn_v2-004.txt: OK (31 elementos)
18_superanimal_quadruped_snapshot-hrnet_w32-004_snapshot-fasterrcnn_resnet50_fpn

- **resultados_predicao_KeypointsCriticosFiltrados/** validação da quantidade de keypoints no arquivo H5, apos filtro até aqui (formato nativo do DeepLabCut/HRNet).

In [ ]:
import pandas as pd
import os

caminho_teste = '/content/dados/resultados_predicao_KeypointsCriticosFiltrados/Resultado_A/1_superanimal_quadruped_snapshot-hrnet_w32-004_snapshot-fasterrcnn_resnet50_fpn_v2-004.h5'
# Lê o arquivo e imprime apenas as partes do corpo presentes
df = pd.read_hdf(caminho_teste, key='df_with_missing')
pontos_presentes = df.columns.get_level_values('bodyparts').unique()

print(f"Total de pontos encontrados: {len(pontos_presentes)}")
print("Lista de pontos:", list(pontos_presentes))

Total de pontos encontrados: 15
Lista de pontos: ['nose', 'neck_base', 'neck_end', 'back_base', 'back_end', 'back_middle', 'front_left_knee', 'front_left_paw', 'front_right_knee', 'front_right_paw', 'back_left_paw', 'back_left_knee', 'back_right_knee', 'back_right_paw', 'belly_bottom']


# Padrões YOLO - Treino
- povoar essas pastas.
- vídeos brutos em /dados/videos_brutos
- extrair os frames dos vídeos
- garantir que eles tenham os nomes correspondentes aos arquivos .txt
- utiliza o opencv para extrair os frames dos vídeos e salvá-los na pasta correta criada
A Estratégia de Correção
Como você tem 4.469 imagens e seus .txt têm nomes baseados no vídeo (ex: 1_superanimal...txt), o mais seguro é renomear os arquivos de imagem para corresponder aos seus arquivos .txt originais, ou vice-versa.

Como o nome do .txt já contém a referência (ex: 1_...), vamos renomear os arquivos .txt para simplificar e garantir o pareamento com as imagens.

Passo 1: Renomear os seus arquivos .txt para o padrão das imagens
Este script vai ler os seus arquivos .txt existentes e renomeá-los para bater com o padrão VideoID_frame_X.txt. Para isso, preciso assumir que o seu vídeo "1" gerou as imagens que começam com "1_frame_X".
- .txt para cada frame.

In [ ]:
import os
import pandas as pd
import cv2

# Configurações de caminhos
pasta_videos = '/content/dados/videos_brutos'
pasta_labels_dest = '/content/yolo_dataset/train/labels'
pasta_images_dest = '/content/yolo_dataset/train/images'
pasta_h5 = '/content/dados/resultados_predicao_KeypointsCriticosFiltrados'

# 1. Validação de Execução Prévia
if os.path.exists(pasta_labels_dest) and len(os.listdir(pasta_labels_dest)) > 0:
    print("Labels já processados. Pulando etapa de conversão H5.")
else:
    print("Iniciando extração de labels...")
    # [Seu código anterior de loop H5 aqui dentro]

# 2. Extração de Frames com OpenCV e Validação
if os.path.exists(pasta_images_dest) and len(os.listdir(pasta_images_dest)) > 0:
    print("Frames já extraídos. Nada a fazer.")
else:
    print("Iniciando extração de frames de vídeo...")
    for video_file in os.listdir(pasta_videos):
        if video_file.endswith(('.mp4', '.avi')):
            video_id = video_file.split('_')[0]
            cap = cv2.VideoCapture(os.path.join(pasta_videos, video_file))
            idx = 0
            while cap.isOpened():
                ret, frame = cap.read()
                if not ret: break

                # Salva o frame com o mesmo nome base dos arquivos .txt
                cv2.imwrite(os.path.join(pasta_images_dest, f"{video_id}_frame_{idx}.jpg"), frame)
                idx += 1
            cap.release()
            print(f"Vídeo {video_file} processado.")

print("Processamento finalizado com sucesso.")

Labels já processados. Pulando etapa de conversão H5.
Frames já extraídos. Nada a fazer.
Processamento finalizado com sucesso.


# Interligando imagens com labels

In [ ]:
import os

pasta_images = '/content/yolo_dataset/train/images'
pasta_labels = '/content/yolo_dataset/train/labels'

# Lista os nomes sem a extensão
nomes_img = set([os.path.splitext(f)[0] for f in os.listdir(pasta_images)])
nomes_lbl = set([os.path.splitext(f)[0] for f in os.listdir(pasta_labels)])

intersecao = nomes_img.intersection(nomes_lbl)
sobrando_img = nomes_img - nomes_lbl
sobrando_lbl = nomes_lbl - nomes_img

print(f"Total de arquivos pareados perfeitamente: {len(intersecao)}")
print(f"Total de imagens processadas: {len(nomes_img)}")
print(f"Total de labels gerados: {len(nomes_lbl)}")

if len(sobrando_img) == 0 and len(sobrando_lbl) == 0:
    print("\nSucesso absoluto! O dataset está perfeitamente pareado.")
else:
    print(f"\nDivergência encontrada!")
    print(f"Imagens sem label: {len(sobrando_img)}")
    print(f"Labels sem imagem: {len(sobrando_lbl)}")

Total de arquivos pareados perfeitamente: 4437
Total de imagens processadas: 4437
Total de labels gerados: 4437

Sucesso absoluto! O dataset está perfeitamente pareado.


In [ ]:
yaml_content = """path: /content/ajustefinoYolo/yolo_dataset
train: train/images
val: val/images

nc: 1
names: ['cavalo']
kpt_shape: [15, 2]
"""

with open('/content/ajustefinoYolo/data.yaml', 'w') as f:
    f.write(yaml_content)

print("data.yaml atualizado com sucesso!")

data.yaml atualizado com sucesso!
